In [ ]:
from datasets import load_dataset
from collections import Counter
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import numpy as np
import json

In [34]:
dataset = load_dataset("bigbio/bc5cdr")

/Users/Usuario/Desktop/Studium/MLT SoSe26/Information Extraction/Med_IE_System/.venv/lib/python3.13/site-packages/datasets/load.py:1491: FutureWarning: The repository for bigbio/bc5cdr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/bc5cdr
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['passages'],
        num_rows: 500
    })
    test: Dataset({
        features: ['passages'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['passages'],
        num_rows: 500
    })
})


In [ ]:
example = dataset["train"][0]

print(json.dumps(example, indent=2))

{
  "passages": [
    {
      "document_id": "227508",
      "type": "title",
      "text": "Naloxone reverses the antihypertensive effect of clonidine.",
      "entities": [
        {
          "id": "0",
          "offsets": [
            [
              0,
              8
            ]
          ],
          "text": [
            "Naloxone"
          ],
          "type": "Chemical",
          "normalized": [
            {
              "db_name": "MESH",
              "db_id": "D009270"
            }
          ]
        },
        {
          "id": "1",
          "offsets": [
            [
              49,
              58
            ]
          ],
          "text": [
            "clonidine"
          ],
          "type": "Chemical",
          "normalized": [
            {
              "db_name": "MESH",
              "db_id": "D003000"
            }
          ]
        }
      ],
      "relations": [
        {
          "id": "R0",
          "type": "CID",
          "arg1_id": "

Type values: title, abstract
Fields: entities, relations, text
CID - Chemical-Induced Disease
arg1_id: Chemical ID
arg2_id: Disease ID
Relation: arg1_id induced arg2_id
Offsets: Character indices
MeSH: database with standardized names and IDs for medical concepts

In [ ]:
chemical_count = 0
disease_count = 0

for doc in dataset["train"]:
    for passage in doc["passages"]:
        for entity in passage["entities"]:
            if entity["type"] == "Chemical":
                chemical_count += 1
            if entity["type"] == "Disease":
                disease_count += 1

print(f"Chemical: {chemical_count}")
print(f"Disease: {disease_count}")

Chemical: 5207
Disease: 4363


In [17]:
chem_no_mesh = 0
dis_no_mesh = 0

for doc in dataset["train"]:
    for passage in doc["passages"]:
        for entity in passage["entities"]:
            if not entity["normalized"]:
                if entity["type"] == "Chemical":
                    chem_no_mesh += 1
                if entity["type"] == "Disease":
                    dis_no_mesh += 1

print(f"Missing MeSH for Chemical: {chem_no_mesh}")
print(f"Missing MeSH for Disease: {dis_no_mesh}")

Missing MeSH for Chemical: 44
Missing MeSH for Disease: 32


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
token_counts = []

for doc in dataset["train"]:
    full_text = ""
    for passage in doc["passages"]:
        full_text += passage["text"] + " "
    token_counts.append(len(tokenizer(full_text)["input_ids"]))

print(f"Most tokens in a doc: {max(token_counts)}")
print(f"Mean of the token count: {np.mean(token_counts)}")
print(f"Docs longer than 512 tokens: {sum(1 for count in token_counts if count > 512)}")


Most tokens in a doc: 672
Mean of the token count: 257.242
Docs tolnger than 512 tokens: 18
